# Pearls AQI Predictor — Exploratory Data Analysis

Analysis of historical AQI, pollutant, and weather trends for Islamabad, using the combined dataset (2-year historical backfill + live hourly readings) stored in the Hopsworks Feature Store.

**Sections:**
1. Data loading & overview
2. AQI trend over time
3. AQI distribution & category breakdown
4. Seasonal patterns (monthly, day-of-week)
5. Correlation between pollutants/weather and AQI
6. Weather vs AQI relationships
7. Pollutant trends
8. Summary of findings

In [ ]:
import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hopsworks
from dotenv import load_dotenv

sys.path.append(os.path.join(os.getcwd(), ".."))
from utils.city_config import CITY_NAME

load_dotenv()

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

AQI_CATEGORY_COLORS = {
    "Good": "#00E400",
    "Moderate": "#FFFF00",
    "Unhealthy for Sensitive Groups": "#FF7E00",
    "Unhealthy": "#FF0000",
    "Very Unhealthy": "#8F3F97",
    "Hazardous": "#7E0023",
}

def categorize_aqi(v):
    if v <= 50: return "Good"
    if v <= 100: return "Moderate"
    if v <= 150: return "Unhealthy for Sensitive Groups"
    if v <= 200: return "Unhealthy"
    if v <= 300: return "Very Unhealthy"
    return "Hazardous"

## 1. Data Loading & Overview

Connects to Hopsworks, reads the full `aqi_features` Feature Group, and aggregates hourly/daily rows into one row per calendar day (same methodology used throughout the project's training pipeline).

In [ ]:
project = hopsworks.login(
    project=os.getenv("HOPSWORKS_PROJECT"),
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value=os.getenv("HOPSWORKS_API_KEY"),
)
fs = project.get_feature_store()
fg = fs.get_feature_group("aqi_features", version=1)
raw_df = fg.read()
raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"])
print(f"Raw rows: {len(raw_df)}")

In [ ]:
raw_df["date"] = raw_df["timestamp"].dt.date
agg_cols = [
    "aqi", "pm25_median", "pm10_median", "o3_median", "no2_median",
    "so2_median", "co_median", "temperature_median", "humidity_median",
    "pressure_median", "wind_speed_median",
]
df = raw_df.groupby("date")[agg_cols].median().reset_index()
df["timestamp"] = pd.to_datetime(df["date"])
df["month"] = df["timestamp"].dt.month
df["month_name"] = df["timestamp"].dt.month_name()
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["day_name"] = df["timestamp"].dt.day_name()
df["aqi_category"] = df["aqi"].apply(categorize_aqi)
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Daily rows: {len(df)}")
print(f"Date range: {df['timestamp'].min().date()} to {df['timestamp'].max().date()}")
df.head()

In [ ]:
print("Missing values per column:")
print(df[agg_cols].isna().sum())
print("\nSummary statistics:")
df[agg_cols].describe()

## 2. AQI Trend Over Time

Raw daily AQI plus a 7-day rolling average to smooth out day-to-day noise and reveal the underlying trend.

In [ ]:
df["rolling_7d"] = df["aqi"].rolling(7, min_periods=1).mean()

fig, ax = plt.subplots()
ax.plot(df["timestamp"], df["aqi"], alpha=0.3, label="Daily AQI", color="gray")
ax.plot(df["timestamp"], df["rolling_7d"], label="7-day rolling average", color="#2E7D5B", linewidth=2)
for lo, hi, color in [(150, 200, "#FF0000"), (200, 300, "#8F3F97"), (300, 500, "#7E0023")]:
    ax.axhspan(lo, hi, color=color, alpha=0.05)
ax.set_title(f"Daily AQI Trend — {CITY_NAME}")
ax.set_xlabel("Date")
ax.set_ylabel("AQI")
ax.legend()
plt.tight_layout()
plt.show()

## 3. AQI Distribution & Category Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["aqi"], bins=30, color="#2E7D5B", edgecolor="white")
axes[0].set_title("AQI Distribution")
axes[0].set_xlabel("AQI")
axes[0].set_ylabel("Number of days")

category_counts = df["aqi_category"].value_counts()
colors = [AQI_CATEGORY_COLORS[c] for c in category_counts.index]
axes[1].pie(category_counts, labels=category_counts.index, autopct="%1.0f%%", colors=colors, textprops={"fontsize": 9})
axes[1].set_title("Share of Days by AQI Category")

plt.tight_layout()
plt.show()

print(category_counts)

## 4. Seasonal Patterns

How AQI varies by month and by day of the week.

In [ ]:
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]

fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(data=df, x="month_name", y="aqi", order=month_order, ax=ax, color="#2E7D5B")
ax.set_title("AQI by Month")
ax.set_xlabel("")
ax.set_ylabel("AQI")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="day_name", y="aqi", order=day_order, ax=ax, color="#3B7DD8")
ax.set_title("AQI by Day of Week")
ax.set_xlabel("")
ax.set_ylabel("AQI")
plt.tight_layout()
plt.show()

## 5. Correlation Between Pollutants, Weather, and AQI

In [ ]:
corr_cols = agg_cols
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdYlGn_r", center=0, ax=ax, square=True)
ax.set_title("Correlation Heatmap: Pollutants, Weather, and AQI")
plt.tight_layout()
plt.show()

print("Correlation with AQI, sorted:")
print(corr_matrix["aqi"].sort_values(ascending=False))

## 6. Weather vs AQI Relationships

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(data=df, x="temperature_median", y="aqi", ax=axes[0,0], alpha=0.5, color="#D64545")
axes[0,0].set_title("Temperature vs AQI")

sns.scatterplot(data=df, x="humidity_median", y="aqi", ax=axes[0,1], alpha=0.5, color="#3B7DD8")
axes[0,1].set_title("Humidity vs AQI")

sns.scatterplot(data=df, x="wind_speed_median", y="aqi", ax=axes[1,0], alpha=0.5, color="#2E7D5B")
axes[1,0].set_title("Wind Speed vs AQI")

sns.scatterplot(data=df, x="pressure_median", y="aqi", ax=axes[1,1], alpha=0.5, color="#8F3F97")
axes[1,1].set_title("Pressure vs AQI")

plt.tight_layout()
plt.show()

## 7. Pollutant Trends Over Time

In [ ]:
pollutants = ["pm25_median", "pm10_median", "o3_median", "no2_median", "so2_median", "co_median"]

fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)
for ax, col in zip(axes.flatten(), pollutants):
    ax.plot(df["timestamp"], df[col], color="#2E7D5B", linewidth=1)
    ax.set_title(col.replace("_median", "").upper())
    ax.set_ylabel("Concentration")

plt.tight_layout()
plt.show()

In [ ]:
dominant_pollutant = df[pollutants].corrwith(df["aqi"]).sort_values(ascending=False)
print("Correlation of each pollutant with AQI (highest = most dominant driver):")
print(dominant_pollutant)

## 8. Summary of Findings

Fill in after running the notebook on your actual data — example structure:

- **Dominant pollutant:** Whichever pollutant shows the highest correlation with AQI in Section 7 (expected: PM2.5, consistent with how the target `aqi` column itself is derived).
- **Seasonality:** Note which months show the worst AQI (commonly winter months for Islamabad, due to temperature inversion trapping pollutants).
- **Weather relationships:** Note the sign/strength of correlations from Section 5 — e.g. does higher wind speed correlate with lower AQI (dispersion effect)? Does higher humidity correlate with higher AQI (particulate binding)?
- **Day-of-week effect:** Note whether weekdays vs weekends show meaningfully different AQI (traffic/industrial activity patterns).
- **Data quality:** Note any gaps or anomalies observed in Section 1's missing-value check.